In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

%matplotlib inline

<h1>Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Data loading" data-toc-modified-id="Data loading-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Data loading</a></span></li><li><span><a href="#Matrix multiplication" data-toc-modified-id="Matrix-multiplication-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Matrix-multiplication</a></span></li><li><span><a href="#Transformation-algorithm" data-toc-modified-id="Transformation-algorithm-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Conversion algorithm</a></span></li><li><span><a href="#Checking-algorithm" data-toc-modified-id="Checking-algorithm-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Checking algorithm</a></span></li><li><span><a href="#Now-we-implement-our-own-algorithm." data-toc-modified-id="Now-we-implement-our-own-algorithm.-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Now we implement our own algorithm.</a></span></li>

# Protection of personal data of clients

We need to protect the data of clients of the insurance company "Though the Flood". We will develop a method for converting data so that it is difficult to recover personal information from it.

It is necessary to protect the data so that the quality of machine learning models does not deteriorate during conversion.

## Loading data

Let's load the data and see if there are gaps or if the column data is cast to the wrong type.

In [2]:
try:
    data = pd.read_csv('/datasets/insurance.csv')

except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/insurance.csv')

data = data.rename(columns={
    '\u041f\u043e\u043b': 'gender',
    '\u0412\u043e\u0437\u0440\u0430\u0441\u0442': 'age',
    '\u0417\u0430\u0440\u043f\u043b\u0430\u0442\u0430': 'salary',
    '\u0427\u043b\u0435\u043d\u044b \u0441\u0435\u043c\u044c\u0438': 'family_members',
    '\u0421\u0442\u0440\u0430\u0445\u043e\u0432\u044b\u0435 \u0432\u044b\u043f\u043b\u0430\u0442\u044b': 'insurance_payouts',
})
data.info()
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender                5000 non-null   int64  
 1   age            5000 non-null   float64
 2   salary           5000 non-null   float64
 3   family_members        5000 non-null   int64  
 4   insurance_payouts  5000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 195.4 KB


,gender,age,salary,family_members,insurance_payouts
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.499000,30.952800,39916.360000,1.194200,0.148000
std,0.500049,8.440807,9900.083569,1.091387,0.463183
min,0.000000,18.000000,5300.000000,0.000000,0.000000
25%,0.000000,24.000000,33300.000000,0.000000,0.000000
50%,0.000000,30.000000,40200.000000,1.000000,0.000000
75%,1.000000,37.000000,46600.000000,2.000000,0.000000
max,1.000000,65.000000,79000.000000,6.000000,5.000000


Lucky, the data is in perfect order and there are no gaps!

## Matrix multiplication

Designations:

- $X$ — matrix of features (the zero column consists of ones)

- $y$ — vector of the target feature

- $P$ is the matrix by which the features are multiplied

- $w$ — vector of linear regression weights (zero element equals shift)

Predictions:

$$
a = Xw
$$

Learning Objective:

$$
w = \arg\min_w MSE(Xw, y)
$$

Training formula:

$$
w = (X^T X)^{-1} X^T y
$$

Let's check whether the quality of linear regression changes when multiplying features by an invertible matrix.

###### Step 1. Enter the Z matrix.
Let Z be our new matrix $$Z = XP$$
<br></br>
###### Step 2. Let's derive the updated formula for prediction
$$a1 = Zw = Z (Z^T Z)^{-1} Z^T y $$
###### Step 3. Expand our Z matrix
$$a1 = XP ((XP)^{T} XP))^{-1} (XP)^T y$$
###### Step 4. Transform the expressions $$(XP)^T = P^T X^T$$ and simplify
$$a1 = XP(P^TX^TXP)^{-1} P^TX^T y$$
$$a1 = XP (P)^{-1} (X^TX)^{-1} (P^T)^{-1} P^T X^T y$$
we reduce expressions that are inverse to each other
$$a1 = X(X^T X)^{-1} X^T y $$
###### Step 4. Compare
$$a1 = a$$
since
$$(X^T X)^{-1} X^T y = w$$
by condition and $$Xw = X(X^T X)^{-1} X^T y$$
The training formula has not changed - the quality of linear regression will not change.

## Conversion algorithm

**Algorithm**

An encryption algorithm, the essence of which is to multiply a matrix of features by an invertible matrix.
<br></br>
- We generate and fix a random square matrix as an encryption key with a size equal to the number of features, not taking into account the target one.
<br></br>
- We scalarly multiply the feature matrix by our invertible square matrix obtained in the first step.
<br></br>
- We use the resulting product as features to split the data and train the model.

**Rationale**

We multiply each set of features from features by identical sets of vectors (column - vector) from a square invertible matrix and write them into a new vector - row. Accordingly, the final weights of each such vector will be equal to the weights of the set of features.

In Point 2 we have already proven why the quality will not decrease - the expressions are identical.

## Algorithm check

Let us denote the target and non-target features. As we know, the target attribute is *Insurance payments*.

In [3]:
features = data.drop('insurance_payouts', axis=1)
target = data['insurance_payouts']

Let's split the data into training and test samples in a ratio of 3:1.

In [4]:
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2,
                                                                            random_state=12345)

Let's train the model and calculate the *r2 score* metric.

In [5]:
from sklearn.metrics import r2_score

model = LinearRegression()
model.fit(features_train, target_train)
predictions = model.predict(features_test)
print('R2 score = ', np.round(r2_score(y_true=target_test, y_pred=predictions), 5))

R2 score =  0.41177


## Now let's implement our own algorithm.

Let's create a square invertible matrix using the np.random.normal method; when using this method, the probability of obtaining an irreversible matrix is ​​close to zero.
In size we will pass the size of the features - 1, since one of them is target.

In [6]:
square_matrix = np.random.normal(size=(data.shape[1] - 1, data.shape[1] - 1))
square_matrix

array([[-0.12954955,  0.59662364,  0.1716113 , -0.9852623 ],
       [ 0.26223839,  1.0747992 ,  0.28864125, -0.12068828],
       [ 1.45375467, -0.25287972, -0.05163723,  0.78913095],
       [-0.87614381,  1.52586754, -0.34181388, -0.34817173]])

Here we will check that we get the identity matrix when multiplying a matrix by its inverse.

In [7]:
inverse_square_matrix = np.linalg.inv(square_matrix)
square_matrix @ inverse_square_matrix

array([[ 1.00000000e+00, -1.79310772e-17,  3.16241951e-17,
         3.14255638e-17],
       [ 3.67417878e-17,  1.00000000e+00,  1.31002022e-16,
         1.21128125e-16],
       [-3.02125209e-16,  1.02331483e-17,  1.00000000e+00,
        -2.95836171e-18],
       [ 7.77510070e-17,  7.98418285e-17,  2.09903577e-17,
         1.00000000e+00]])

Let's create our encrypted features by scalar multiplication by an invertible matrix.

In [8]:
new_features = features @ square_matrix

We break down the data.

In [9]:
new_features_train, new_features_test, new_target_train, new_target_test = train_test_split(new_features, target,
                                                                                            test_size=0.2,
                                                                                            random_state=12345)

And finally we calculate the metric.

In [10]:
new_model = LinearRegression()
new_model.fit(new_features_train, new_target_train)
new_predictions = new_model.predict(new_features_test)
print('R2 score = ', np.round(r2_score(y_true=new_target_test, y_pred=new_predictions), 5))

R2 score =  0.41177


The Linear Regression metric on the original data and the metric on the encrypted data are equal. Therefore, the transformation algorithm did not affect model quality.

This project involved reviewing and analyzing data. Lucky - they were in perfect order without gaps and with correct type casting. Next, the question was investigated, which was to multiply the features by an invertible matrix, would the quality of the linear regression change? We received a negative answer to the question, proving that in the usual and in our question the learning formulas are identical, which indicates that the quality of linear regression will not change, but the data will be encrypted. Next, we used this algorithm and applied it in practice with our data: we checked it on Ordinary Linear Regression and taking into account the multiplication of features by an invertible matrix, and as it turned out, the R2 Score metric is the same and amounts to 0.41177 (without taking into account the selection of hyperparameters).